In [ ]:
import pandas as pd
import numpy as np
import pgeocode
from matplotlib import pyplot as plt
from pathlib import Path

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

STRESS_TREND_IMAGE = PROJECT_ROOT / "outputs" / "plots" / "mlp_stress_trend_over_time.png"
HEATMAP_IMAGE = PROJECT_ROOT / "outputs" / "plots" / "mlp_zip_heatmap_top15.png"
TOP_TRENDS_IMAGE = PROJECT_ROOT / "outputs" / "plots" / "mlp_top10_zip_trends.png"

In [ ]:
nomi = pgeocode.Nominatim('us')

results_with_meta = pd.read_csv(PROJECT_ROOT / "outputs" / "mlp_results_with_meta.csv", dtype={"zip_code": str})
results_with_meta["zip_code"] = results_with_meta["zip_code"].astype(str).str.zfill(5)

# Build ZIP-month summaries with metadata
daily_results = results_with_meta.copy()
daily_results["date"] = pd.to_datetime(daily_results["date"])
daily_results["month"] = daily_results["date"].dt.to_period("M").dt.to_timestamp()

monthly = (
    daily_results.groupby(["zip_code", "month"], as_index=False)
    .agg(
        mean_pred_prob=("pred_prob", "mean"),
        pred_high_stress_rate=("pred_label", "mean"),
        n_obs=("pred_label", "size"),
    )
    .sort_values(["zip_code", "month"])
)

# Enrich with ZIP metadata (place name, state, county)
zip_codes = sorted(monthly["zip_code"].unique())
zip_info = []
for zc in zip_codes:
    row = nomi.query_postal_code(zc)
    place_name = row.place_name if isinstance(row.place_name, str) else zc
    state = row.state_code if isinstance(row.state_code, str) else ""
    county = row.county_name if isinstance(row.county_name, str) else ""
    zip_info.append({"zip_code": zc, "area_label": f"{zc} - {place_name}", "state": state, "county": county})
zip_info = pd.DataFrame(zip_info)

monthly = monthly.merge(zip_info, on="zip_code", how="left")

# First-last summary for ranking
first_last = (
    monthly.groupby("zip_code", as_index=False)
    .agg(
        first_date=("month", "first"),
        last_date=("month", "last"),
        first_mean_pred_prob=("mean_pred_prob", "first"),
        last_mean_pred_prob=("mean_pred_prob", "last"),
    )
    .assign(delta_mean_pred_prob=lambda df: df["last_mean_pred_prob"] - df["first_mean_pred_prob"])
    .merge(zip_info, on="zip_code", how="left")
    .sort_values("delta_mean_pred_prob", ascending=False)
)

# Boston-core ZIPs (MA + Suffolk)
core_monthly = monthly[(monthly["state"] == "MA") & (monthly["county"] == "Suffolk")].copy()
core_change = first_last[(first_last["state"] == "MA") & (first_last["county"] == "Suffolk")].copy()

In [ ]:
# Aggregate stress trend over time
date_trend = monthly.groupby("month", as_index=False).agg(
    mean_pred_prob=("mean_pred_prob", "mean"),
    pred_high_stress_rate=("pred_high_stress_rate", "mean"),
).sort_values("month")

plt.figure(figsize=(14, 6.5))
plt.plot(date_trend["month"], date_trend["mean_pred_prob"], marker="o", linewidth=2.5, label="Mean predicted probability")
plt.plot(date_trend["month"], date_trend["pred_high_stress_rate"], marker="o", linewidth=2.5, label="Predicted high-stress rate")
plt.title("MLP Stress Signals Over Time")
plt.xlabel("Date")
plt.ylabel("Share / probability")
plt.ylim(0, 1)
plt.legend(loc="upper left")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(STRESS_TREND_IMAGE)
plt.show()

In [ ]:
# Top 10 Boston-core ZIPs - trend lines
top10 = core_change.sort_values("delta_mean_pred_prob", ascending=False).head(10).copy()
top10_zips = top10["zip_code"].tolist()
top10_labels = top10.set_index("zip_code").loc[top10_zips, "area_label"].tolist()
zip_to_label = dict(zip(top10_zips, top10_labels))

top10_monthly = core_monthly[core_monthly["zip_code"].isin(top10_zips)].copy()
top10_monthly["label"] = top10_monthly["zip_code"].map(zip_to_label)

fig, ax = plt.subplots(figsize=(20, 8))
for label, group in top10_monthly.sort_values("month").groupby("label"):
    ax.plot(group["month"], group["mean_pred_prob"], linewidth=2.5, label=label)
ax.set_title("Top 10 Boston-Core ZIP Areas (MA Suffolk) with Largest Stress Increase")
ax.set_xlabel("Date")
ax.set_ylabel("Mean predicted stress probability")
ax.set_ylim(0, 1)
ax.legend(title="ZIP - Area", ncol=2, loc="upper left")
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(TOP_TRENDS_IMAGE)
plt.show()